# Minimal COVFEE LLM Pipeline

Fill the first code cell, run the preprocessing/prompt cells, manually place LLM outputs as `data.json`, then run evaluation.

The first cell is an example configuration for a single recording. The time offsets are manually determined, but the videos provided in the dataset have embedded timestamps that can be used to automatically determine the offsets. The problem I had was that I took clips from the original video, but the timestamps in the clips metadata were the same as original video. So I had to manually determine the offsets for each clip. The offsets are relative to the original video, and the annotation times are relative to the clip. The `source_video_start_abs_time` is the absolute time of the start of the original video, which can be used to convert the annotation times to absolute times and can be automatically determined from the video metadata.

| video name | GH040226        | GH040336       |   |   |
|------------|-----------------|----------------|---|---|
| p_id       | 01,03,16,17,24, | 03,16,18,20,27 |   |   |
| clip time  | 5:30-10:30(v1)  | 3:00-8:00(v2)  |   |   |
|            |                 |                |   |   |

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from llm_pipeline import RecordingConfig, build_manual_labels_from_covfee, write_prompt_bundle, evaluate_llm_predictions

# Required recording identity.
ANNOTATION_ID = "24_v1"
PARTICIPANT_NO = 24
MIDGE_ID = 65
VIDEO_NAME = "v1"

# Required absolute input paths.
ANNOTATION_JSON = Path(r"G:\\TUD_CESE\\OneDrive - Delft University of Technology\\Smart Cup\\relativeWorks\\llm_pipeline\\input\\24_v1.json")
ACC_PATHS = [
    Path(r"G:\\TUD_CESE\\OneDrive - Delft University of Technology\\Smart Cup\\relativeWorks\\llm_pipeline\\input\\ACC_0.csv"),
]
VIDEO_PATH = Path(r"G:\\TUD_CESE\\OneDrive - Delft University of Technology\\Smart Cup\\relativeWorks\\llm_pipeline\\input\\v1.mp4")

# Manual alignment. Annotation times are clip-relative; clip offsets are relative to the source video GH040226.mp4.
SOURCE_VIDEO_START_ABS_TIME = "2025-07-17 13:57:00.166833333"
CLIP_START_OFFSET_S = 5 * 60 + 30
CLIP_END_OFFSET_S = 10 * 60 + 30

# Experiment settings.
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
TARGET_FS = 50.0
WINDOW_SECONDS = 10.0
N_LLM_RUNS = 4
REFERENCE_TRAIN_RATIO = 0.70
FEW_SHOT_CANDIDATE_START_RATIO = 0.0


## 1. Build Sample-Level Manual Labels

In [2]:
recording = RecordingConfig(
    annotation_id=ANNOTATION_ID,
    participant_no=PARTICIPANT_NO,
    midge_id=MIDGE_ID,
    video_name=VIDEO_NAME,
    annotation_json=ANNOTATION_JSON,
    acc_paths=ACC_PATHS,
    video_path=VIDEO_PATH,
    source_video_start_abs_time=SOURCE_VIDEO_START_ABS_TIME,
    clip_start_offset_s=CLIP_START_OFFSET_S,
    clip_end_offset_s=CLIP_END_OFFSET_S,
    target_fs=TARGET_FS,
)

manual_result = build_manual_labels_from_covfee(recording, OUTPUT_ROOT)

print("manual_csv:", manual_result.manual_csv)
print("config_json:", manual_result.config_json)
print("summary_json:", manual_result.summary_json)
manual_result.manual_df["manual_label"].value_counts().sort_index()


manual_csv: G:\TUD_CESE\OneDrive - Delft University of Technology\Smart Cup\relativeWorks\llm_pipeline\outputs\manual_labels\manual_labels_24_v1.csv
config_json: G:\TUD_CESE\OneDrive - Delft University of Technology\Smart Cup\relativeWorks\llm_pipeline\outputs\manual_labels\recording_config_24_v1.json
summary_json: G:\TUD_CESE\OneDrive - Delft University of Technology\Smart Cup\relativeWorks\llm_pipeline\outputs\manual_labels\manual_label_summary_24_v1.json
overlay_png: G:\TUD_CESE\OneDrive - Delft University of Technology\Smart Cup\relativeWorks\llm_pipeline\outputs\figs\24_v1\24_v1_manual_overlay.png


manual_label
Drinking     134
Gesture     3286
Nodding     2297
Still       9201
Toasting      82
Name: count, dtype: int64

## 2. Generate LLM Prompts and Input JSON

In [3]:
prompt_bundle = write_prompt_bundle(
    manual_result.manual_df,
    OUTPUT_ROOT,
    ANNOTATION_ID,
    n_runs=N_LLM_RUNS,
    window_seconds=WINDOW_SECONDS,
    sampling_rate_hz=TARGET_FS,
    reference_train_ratio=REFERENCE_TRAIN_RATIO,
    few_shot_candidate_start_ratio=FEW_SHOT_CANDIDATE_START_RATIO,
)

print("window_index_csv:", prompt_bundle.window_index_csv)
for method_id, paths in prompt_bundle.method_artifacts.items():
    print("\n", method_id)
    print("  prompt_txt:", paths["prompt_txt"])
    print("  input_json:", paths["input_json"])
    print("  prediction_dir:", paths["prediction_dir"])


window_index_csv: G:\TUD_CESE\OneDrive - Delft University of Technology\Smart Cup\relativeWorks\llm_pipeline\outputs\input_json\24_v1\window_index_24_v1.csv

 llm_zero_shot_xyz_timeline
  prompt_txt: G:\TUD_CESE\OneDrive - Delft University of Technology\Smart Cup\relativeWorks\llm_pipeline\outputs\prompts\24_v1\llm_zero_shot_xyz_timeline\prompt.txt
  input_json: G:\TUD_CESE\OneDrive - Delft University of Technology\Smart Cup\relativeWorks\llm_pipeline\outputs\input_json\24_v1\llm_zero_shot_xyz_timeline\input.json
  prediction_dir: G:\TUD_CESE\OneDrive - Delft University of Technology\Smart Cup\relativeWorks\llm_pipeline\outputs\llm_predictions\24_v1\llm_zero_shot_xyz_timeline

 llm_few_shot_xyz_timeline
  prompt_txt: G:\TUD_CESE\OneDrive - Delft University of Technology\Smart Cup\relativeWorks\llm_pipeline\outputs\prompts\24_v1\llm_few_shot_xyz_timeline\prompt.txt
  input_json: G:\TUD_CESE\OneDrive - Delft University of Technology\Smart Cup\relativeWorks\llm_pipeline\outputs\input_json

## 3. Manual LLM Interaction

For each method, send `prompt.txt` and `input.json` to the LLM. Save each independent response as:

`outputs/llm_predictions/<annotation_id>/<method>/run_01/data.json`

Repeat for `run_02`, `run_03`, and `run_04`.

## 4. Evaluate LLM Outputs

In [4]:
evaluation = evaluate_llm_predictions(
    manual_label_csv=manual_result.manual_csv,
    prediction_root=prompt_bundle.prediction_root,
    output_root=OUTPUT_ROOT,
    annotation_id=ANNOTATION_ID,
    window_index_csv=prompt_bundle.window_index_csv,
    n_runs=N_LLM_RUNS,
    sampling_rate_hz=TARGET_FS,
)

print("figure_dir:", evaluation.figure_dir)
print("model_metrics_csv:", evaluation.model_metrics_csv)
print("model_metrics_png:", evaluation.model_metrics_png)
print("combined_timeline_png:", evaluation.combined_timeline_png)
print("missing_predictions_csv:", evaluation.missing_predictions_csv)
evaluation.summary


figure_dir: G:\TUD_CESE\OneDrive - Delft University of Technology\Smart Cup\relativeWorks\llm_pipeline\outputs\figs\24_v1
model_metrics_csv: G:\TUD_CESE\OneDrive - Delft University of Technology\Smart Cup\relativeWorks\llm_pipeline\outputs\figs\24_v1\24_v1_model_metrics_table.csv
model_metrics_png: G:\TUD_CESE\OneDrive - Delft University of Technology\Smart Cup\relativeWorks\llm_pipeline\outputs\figs\24_v1\24_v1_model_metrics_table.png
combined_timeline_png: G:\TUD_CESE\OneDrive - Delft University of Technology\Smart Cup\relativeWorks\llm_pipeline\outputs\figs\24_v1\24_v1_llm_variants_combined_timeline.png
missing_predictions_csv: G:\TUD_CESE\OneDrive - Delft University of Technology\Smart Cup\relativeWorks\llm_pipeline\outputs\figs\24_v1\24_v1_missing_predictions.csv


,method_id,display_name,run_count,run_ids,macro_precision,macro_precision_std,macro_precision_min,macro_precision_max,macro_recall,macro_recall_std,...,active_accuracy_std,active_accuracy_min,active_accuracy_max,mAP,mAP_std,mAP_min,mAP_max,run_metrics_csv,run_summary_csv,repeated_timeline_png
0,llm_few_shot_xyz_timeline,Few-shot xyz timeline,1,run_01,0.469205,0.0,0.469205,0.469205,0.684254,0.0,...,0.0,0.608381,0.608381,0.258755,0.0,0.258755,0.258755,G:\TUD_CESE\OneDrive - Delft University of Tec...,G:\TUD_CESE\OneDrive - Delft University of Tec...,G:\TUD_CESE\OneDrive - Delft University of Tec...
